# 01 — Data load: raw delivery to tidy store

**Deliverables D2 (timestamp / DST resolution) and D3 (the store builder).**

Turns the 12 delivered Parquet files into `se_interval`: one row per site per
5-minute interval, in the CICCADA convention, partitioned and sorted for querying.

This is the SolarEdge analogue of `build_structured_data.py`. Every conversion happens
here, once:

| | |
|---|---|
| **Time** | naive local civil string (with DST) → `ts_utc` → `ts_aest` (fixed UTC+10) |
| **Sign** | reactive power flipped from the load convention to the generator convention |
| **Units** | W → kW, var → kvar (instantaneous, *not* ×12) |
| **Shape** | per-phase columns folded to site level, phase detail kept separately |
| **Layout** | re-partitioned by AEST month, sorted by `(site_alias, ts_utc)` |

**What it deliberately does not do:** drop implausible values. Nothing is filtered, so
the store reconciles exactly against the raw delivery and every filtering decision stays
visible in the analysis layer where it can be swept.

Run `00_environment_check.ipynb` first.

## Setup

The build takes roughly 30–45 s per month. Peak memory is about the DuckDB limit plus
200 MB, flat across the run, because each month gets its own short-lived connection.

If you are sharing the compute server, set `CICCADA_SE_DUCKDB_MEMORY` and
`CICCADA_SE_DUCKDB_THREADS` before importing. Left unset, DuckDB sizes itself from the
machine. If your raw data sits in OneDrive, consider `CICCADA_SE_STORE_DIR` pointing at
a local scratch disk — the store is ~1.5 GB and rebuilding it would otherwise push the
whole thing through cloud sync each time.

In [ ]:
import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store
from solar_edge.lib import se_ingest as ing

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect(verbose=True)
print("Store directory:", C.STORE_DIR)

## 1. Conventions being applied

Printed before the build, not after, because these are the choices the store bakes in.
From D5 they become part of `manifest()` and travel with every published number.

In [ ]:
display(C.describe_conventions())

## 2. Daylight-saving hazards in the raw delivery

Two windows can break a naive timezone conversion, and they break it in different ways:

- **October (clocks forward)** — local 02:00–02:59 on 2025-10-05 *does not exist* in NSW
  or SA. A row there would be shifted forward onto a real instant and collide with a
  genuine reading. Deduplication would then discard real data.
- **April (clocks back)** — local 02:00–02:59 on 2025-04-06 happens *twice*. A site
  reporting through both passes emits two rows with the same local timestamp.

Neither materialises here: **zero** NSW/SA rows sit in the October gap, and every
April-overlap row has a distinct key. Queensland appears in both windows simply because
it has no daylight saving, so 02:00–03:00 is an ordinary hour there.

The build asserts this per month rather than trusting it — run this against any new
delivery before rebuilding.

In [ ]:
display(ing.dst_hazards(con))

## 3. Build the store

One month at a time. `build_store()` refuses to overwrite an existing store unless told
to; passing `months=[...]` instead appends just those months, which makes an interrupted
build resumable rather than a restart.

Each month writes files named for its source, so a month legitimately spills into the
neighbouring AEST partition without clobbering anything. That spillover is expected: a
South Australian midnight in local time is the previous day in the AEST analysis frame.

In [ ]:
# Set overwrite=True to rebuild from scratch. To resume after an interruption:
#     ing.build_store(con, months=["2025-07", "2025-08"])
stats = ing.build_store(con, overwrite=True)
display(stats)

print(f"Total rows written: {stats.store_rows.sum():,}")
print(f"Duplicate rows removed: {stats.duplicates_removed.sum():,}")
print(f"UTC collisions (must be 0): {stats.utc_collisions.sum()}")
print(f"Wall time: {stats.seconds.sum() / 60:.1f} min")

## 4. Reconciliation

Five claims, each checked against the **deduplicated** raw basis — comparing against raw
totals that still contain duplicates would always show a spurious mismatch.

1. **Rows accounted for** — an identity, not a tolerance: store rows equal raw distinct
   `(site, timestamp)` keys.
2. **Conflicting duplicates** — reported, not asserted (see section 6).
3. **Sites conserved** — 1,602 in both.
4. **Active energy conserved** — exercises the phase folding and the W→kW conversion.
5. **Reactive energy sign-flipped** — the store total must equal −1 × the raw total. This
   is the sign correction stated as a testable claim rather than a comment.

In [ ]:
recon = ing.reconcile(con)
display(recon)

assert recon["pass"].all(), "Reconciliation failed — do not proceed to D4."
print("Reconciliation PASSED.")

## 5. Did the timezone resolution work?

The fleet-wide diurnal profile in the AEST analysis frame is the acceptance test. It
should peak at hour 12 and fall to near zero overnight, with no double-humping or
smearing — which is what you would see if the three states had been pooled in
mismatched frames.

In [ ]:
profile = se_store.q(con, """
    SELECT hour(ts_aest) AS hour_aest,
           count(*)                  AS n_rows,
           round(avg(P_kW), 3)       AS mean_P_kW
    FROM se_interval
    GROUP BY 1 ORDER BY 1
""")

ax = profile.plot(x="hour_aest", y="mean_P_kW", kind="bar", figsize=(11, 3.4),
                  legend=False, color="#e8702a", width=0.85)
ax.set_xlabel("Hour of day (AEST, fixed UTC+10)")
ax.set_ylabel("Mean P (kW)")
ax.set_title("Fleet diurnal profile — peak should sit at hour 12")
peak = int(profile.loc[profile.mean_P_kW.idxmax(), "hour_aest"])
print(f"Peak hour: {peak}  (expected 12)")

### Per-state check

The stronger test. Each state was converted through its own IANA zone, so if that worked
the power-weighted centroids should land on each city's true solar noon expressed in
AEST — Brisbane earliest, then Sydney, then Adelaide about 50 minutes later.

The seasonal comparison is what proves DST was handled: Queensland must barely move
between June and January, while NSW and SA must not shift by a whole hour. If daylight
saving had been ignored, they would.

In [ ]:
centroids = se_store.q(con, """
    SELECT state,
           CASE WHEN month(ts_aest) = 6 THEN 'June' ELSE 'January' END AS season,
           round(sum((hour(ts_aest) * 60 + minute(ts_aest)) * greatest(P_kW, 0))
                 / nullif(sum(greatest(P_kW, 0)), 0) / 60.0, 3) AS centroid_hour_aest
    FROM se_interval
    WHERE month(ts_aest) IN (1, 6)
    GROUP BY 1, 2 ORDER BY 1, 2
""")
display(centroids.pivot(index="state", columns="season", values="centroid_hour_aest"))

## 6. Known data-quality finding: night-time generation

Found while reconciling this step, and worth stating plainly because it would not have
announced itself later.

A small number of rows appear to be timestamped in a **different frame** — most
consistent with UTC — while the rest of the fleet is in local civil time. A UTC-stamped
midday reading lands around 21:00–03:00 in the AEST frame, producing several kW of
"generation" in the middle of the night.

**Measured: 22,124 rows, 0.026% of the store, confined to 20 sites**, peaking at 9.2 kW.
This is also what the conflicting duplicates in section 4 are: mis-framed rows landing on
top of genuine ones.

Two consequences:

- The deduplication rule keeps the **lowest** active power at a collision, which is the
  physically plausible reading when a daytime value has landed on a night timestamp.
- These sites are **not excluded here.** That is an analysis-layer decision for D6/D7
  where it can be swept, not an ingest-layer one that silently drops data.

It matters most for anything touching the overnight envelope — including the night-time
Volt-VAr question that the Ausgrid AMI workstream treats as a differentiator.

In [ ]:
anomaly = ing.night_generation_anomaly(con)
print(f"{len(anomaly)} affected sites, {anomaly.n_night_rows.sum():,} rows "
      f"({100 * anomaly.n_night_rows.sum() / stats.store_rows.sum():.4f}% of the store)")
display(anomaly.head(10))

anomaly.to_csv(C.ARTEFACT_DIR / "night_generation_anomaly_sites.csv", index=False)
print(f"Site list written to {C.ARTEFACT_DIR / 'night_generation_anomaly_sites.csv'}")

## 7. Where the daylight-saving rows landed

Filtered on `ts_utc`, since the transition is an instant. Expect a handful of rows and
essentially no generation — this is 02:00–03:00 local, inside the ~3% overnight coverage.

In [ ]:
display(ing.dst_audit(con))

## 8. Was the re-partitioning worth it?

The delivered files have a single row group each, so any predicate forces a full column
scan. The store is sorted by `(site_alias, ts_utc)` within month partitions, which is
what lets DuckDB skip row groups.

The single-site query is the one to watch — that access pattern drives every per-site
diagnostic and day plot from here on.

In [ ]:
import time

def timed(label, sql):
    start = time.perf_counter()
    n = con.execute(sql).fetchone()[0]
    return {"query": label, "seconds": round(time.perf_counter() - start, 3), "rows": n}

raw_files = C.duckdb_path_list(C.raw_files())
bench = pd.DataFrame([
    timed("RAW  : one site, whole year",
          f"SELECT count(*) FROM read_parquet({raw_files}) WHERE site_alias='AUS765'"),
    timed("STORE: one site, whole year",
          "SELECT count(*) FROM se_interval WHERE site_alias='AUS765'"),
    timed("STORE: one site, one month",
          "SELECT count(*) FROM se_interval WHERE site_alias='AUS765' AND dt_month='2025-06'"),
    timed("STORE: Volt-VAr band, whole year",
          "SELECT count(*) FROM se_interval "
          "WHERE V_max > 240 AND V_max < 253 AND hour(ts_aest) BETWEEN 11 AND 13"),
])
display(bench)
display(se_store.store_status(con)[["logical_name", "exists", "size_mb", "n_rows"]])

## What this establishes

- `se_interval` holds **86,640,968 rows** across 1,602 sites, 2025 in the AEST frame.
- Reconciles exactly against the raw delivery: rows are an identity, active energy agrees
  to ~5e-8 relative, and reactive power is confirmed sign-flipped to ~8e-8.
- Timezone resolution validated at fleet scale — the diurnal profile peaks at hour 12,
  and per-state centroids land on each city's true solar noon.
- Single-site queries are roughly **14× faster** than against the raw files.
- One data-quality finding recorded and quantified: 20 sites with mis-framed timestamps.

## Next: D4 — site dimension and BOM grid points

Builds `se_site`: one row per site with postcode, state, timezone, phase count, an
`s_99` capacity proxy, and the **nearest BOM grid point**.

That last column is the one that matters. It is the only piece Solar Analytics had that
this delivery lacks (`meta_up23c.n_lat` / `n_long`), and without it the D12 irradiance
extract has no join key. `BOM_NCI/Get_ALL_postcodes_ABS.ipynb` already contains the
postcode → grid point machinery. Needs `geopandas` and `shapely`.